
###### 04_gold_layer

###### Purpose

Purpose of this notebook is to focus on to Keep only feature engineering and modeling-ready data and load data into the Gold Delta Table


###### Technologies Used

 -  Databricks

 -  Apache Spark

 -  PySpark

 -  Delta Lake

 - Unity Catalog


###### Input

- Silver Delta Table

######  Output

- Gold Delta Table


######  Architecture

```text


Read Silver Delta Table
    ↓
Feature Engineering
    ↓
Create Model-ready Features
    ↓
Write Gold Delta Table
   ↓
Validate Target Distribution

```


###### Skills Covered

- Feature Engineering

- Delta Lake

- Gold Layer

- Machine Learning Data Preparation

###### Section 0 : Call project config notebook

In [0]:
%run ./00_project_config

###### Section 1 : Read Silver Delta Table

In [0]:
silver_df = spark.table(SILVER_TABLE)
display(silver_df.limit(10))

###### Section 2 : Create Model-Ready Features

In [0]:
gold_df = (
    silver_df
    .withColumn(
        "Churn_Flag",
        when(col("Churn") == "Yes", 1).otherwise(0)
    )
    .withColumn(
        "Tenure_Group",
        when(col("tenure") < 12, "New")
        .when(col("tenure") < 24, "Growing")
        .otherwise("Loyal")
    )
)

###### Section 3 :  Validate both engineered columns

In [0]:

display(
    gold_df.groupBy("Churn_Flag").count().orderBy("Churn_Flag")
)

display(
    gold_df.groupBy("Tenure_Group").count().orderBy("Tenure_Group")
)

In [0]:
expected_tenure_groups = {"New", "Growing", "Loyal"}

actual_tenure_groups = {
    row["Tenure_Group"]
    for row in spark.table(GOLD_TABLE)
    .select("Tenure_Group")
    .distinct()
    .collect()
}

assert actual_tenure_groups == expected_tenure_groups

###### Section 4 : Write Gold Delta Table

In [0]:
gold_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_TABLE)

###### Section 5 : Validate Gold Delta Table

In [0]:

gold_table_df = spark.table(GOLD_TABLE)

print(f"Silver rows: {silver_df.count()}")
print(f"Gold rows: {gold_table_df.count()}")

assert silver_df.count() == gold_table_df.count()

display(gold_table_df.limit(10))

###### Key Learnings

-  Created model-ready features for churn prediction.

-  Converted the churn target into a binary machine-learning label.

-  Created tenure-based customer segments to support predictive modeling.

###### Notebook Conclusion

- Successfully created a Gold Delta table containing feature-engineered, model-ready customer churn data. This dataset will be used in the next notebook for machine learning model training.

###### Next Notebook

05_model_training

Purpose

- Use the Gold Delta table to preprocess features, train multiple churn-classification models, and log their performance.